In [102]:
from langchain_community.document_loaders import RecursiveUrlLoader,WebBaseLoader
from bs4 import BeautifulSoup as soup
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool
import os
from langchain.agents import create_react_agent,AgentExecutor
from langchain import hub
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder,PromptTemplate
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.memory import ConversationBufferWindowMemory

In [7]:
load_dotenv()

True

In [8]:
huggingface_api_key=os.getenv("HF_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")

In [9]:
from huggingface_hub import login
login(token=huggingface_api_key)

In [10]:
# url="https://docs.streamlit.io/"

In [11]:
# loader= RecursiveUrlLoader(url=url,max_depth=5,extractor=lambda x:soup(x,'html.parser').text)

In [12]:
# docs= loader.load()

In [13]:
# text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)
# chunks=text_splitter.split_documents(docs)

In [14]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# vectorstore = FAISS.from_documents(chunks, embedding_model)

In [15]:
# vectorstore.save_local("streamlit_doc_vectors")

In [19]:
vectorstore= FAISS.load_local(folder_path=r"D:\Streamlit Chatbot\streamlit_doc_vectors",embeddings=embedding_model,allow_dangerous_deserialization=True)

In [20]:
retriever = vectorstore.as_retriever()

In [21]:
memory = ConversationBufferWindowMemory(memory_key="chat_history",k=3,return_messages=True)

C:\Users\Shorya Sharma\AppData\Local\Temp\ipykernel_18300\2533680449.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(memory_key="chat_history",k=3,return_messages=True)


In [42]:
prompt=ChatPromptTemplate.from_messages([
      ("system","Answer the user's questions based on the below context:\n\n{context}"),
      MessagesPlaceholder(variable_name="chat_history",optional=True),
      ("human","{input}")
  ])

In [ ]:
llm= ChatGroq(temperature=0.3,groq_api_key="",model_name="mixtral-8x7b-32768")

In [44]:
prompt.pretty_print()

================================ System Message ================================

Answer the user's questions based on the below context:

{context}

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}


In [45]:
docuemnt_chain= create_stuff_documents_chain(llm=llm,prompt=prompt)

In [46]:
rag_chain = create_retrieval_chain(retriever=retriever,combine_docs_chain=docuemnt_chain)

In [47]:
input_text= "Use OpenAI Embdeddings and model."
chat_history= memory.load_memory_variables({})['chat_history']

In [48]:
response=rag_chain.invoke({"input":input_text,"chat_history":chat_history})

In [49]:
print(response['answer'])

Sure! Here's an example of how you can use OpenAI Embeddings and model in your app:

First, you'll need to install the OpenAI Python library if you haven't already:
```
pip install openai
```
Next, you can use the `OpenAI` class from the library to interact with the OpenAI API. Here's an example of how you can use the `create_embedding` method to get embeddings for a given input:

```python
import openai

# Set up OpenAI API key
openai.api_key = "YOUR_API_KEY"

# Create OpenAI client
client = openai.Client()

# Get embeddings for an input
response = client.create_embedding(
  input=[
    "This is an example input",
  ],
  model="text-embedding-ada-002"
)

# Print embeddings
print(response["data"][0]["embedding"])
```
In this example, we're using the `text-embedding-ada-002` model to get embeddings for the input "This is an example input". The `create_embedding` method returns a list of embeddings, one for each input in the `input` parameter.

You can also use the `create_chat` method t

In [50]:
memory.save_context(inputs={"Human":input_text},outputs={"AI":response['answer']})

In [51]:
memory.buffer

[HumanMessage(content='Use OpenAI Embdeddings and model.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='To use OpenAI Embeddings and model in our ChatGPT-like app, we\'ll need to make a few changes to the code. Here\'s an updated version of the code that uses the OpenAI Embeddings and model:\n\n```python\nimport streamlit as st\nfrom openai import OpenAI\n\nst.title("ChatGPT-like clone")\n\n# Set OpenAI API key from Streamlit secrets\nclient = OpenAI(api_key=st.secrets["OPENAI_API_KEY"])\n\n# Set a default model\nif "openai_model" not in st.session_state:\n    st.session_state["openai_model"] = "text-embedding-ada-002"\n\n# Initialize chat history\nif "messages" not in st.session_state:\n    st.session_state.messages = []\n\n# Function to get embeddings from OpenAI\ndef get_embeddings(text):\n    response = client.embeddings.create(\n        model=st.session_state["openai_model"],\n        input=text\n    )\n    return response["data"]\n\n# Function to get response 

# Agentic RAG:

In [119]:
prompt = hub.pull("hwchase17/react")

d:\Streamlit Chatbot\.venv\Lib\site-packages\langsmith\client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [137]:
prompt=PromptTemplate.from_template('''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}

chat_history: {chat_history}''')

In [138]:
prompt.pretty_print()

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}

chat_history: {chat_history}


In [139]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults()

In [140]:
rag_tool= create_retriever_tool(retriever=retriever,name="RAG Pipeline",description= "Having knowledge base of streamlit.")

In [141]:
tools= [rag_tool,search_tool]

In [142]:
agent=create_react_agent(llm=llm,tools=tools,prompt=prompt)

In [143]:
agent_executor= AgentExecutor.from_agent_and_tools(agent=agent,tools=tools,verbose=True,return_intermediate_steps=True,handle_parsing_errors=True)

In [170]:
input_text= "Why we use streamlit?"

In [171]:
response = await agent_executor.ainvoke({"input":input_text,"chat_history":chat_history})



> Entering new AgentExecutor chain...
Thought: The user is asking why we use Streamlit. This question seems to be related to the knowledge base I have access to. I can use the RAG Pipeline to query the knowledge base and find an answer.

Action: RAG Pipeline

Action Input: 'Why do we use Streamlit?'
Streamlit makes it easy for you to visualize, mutate, and share data. The API
reference is organized by activity type, like displaying data or optimizing
performance. Each section includes methods associated with the activity type,
including examples.
Browse our API below and click to learn more about any of our available commands! 🎈
Display almost anything
Write and magic

Prerequisites
As with any programming tool, in order to install Streamlit you first need to make sure your
computer is properly set up. More specifically, you’ll need:

Most Streamlit apps need some kind of data or API access to be useful - either retrieving data to view or saving the results of some user action. This 

In [172]:
print(response['output'])

We use Streamlit because it is a fast and easy way to put a front end on your Python scripts, particularly for data-driven applications. It allows for interactive analysis with sliders, dropdowns, and buttons, and model deployment to share machine learning models as web apps. Streamlit's design allows developers to write applications with minimal boilerplate code, making it quick and easy to create web applications for data science and beyond. Additionally, it provides caching and session state features to improve app performance and create dynamic pages.


In [173]:
memory.save_context(inputs={"Human":input_text},outputs={"AI":response['output']})

In [174]:
memory.buffer

[HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome! If you have any questions or need assistance with anything else, feel free to ask. I'm here to help!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My last words?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='One famous last word is "Die", which was reportedly said by Marie Antoinette, the last queen of France before the French Revolution.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Why we use streamlit?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="We use Streamlit because it is a fast and easy way to put a front end on your Python scripts, particularly for data-driven applications. It allows for interactive analysis with sliders, dropdowns, and buttons, and model deployment to share machine learning models as web apps. Streamlit's design allows developers to write applications with m